In [0]:
#Main libraries:
import pandas as pd
import numpy as np

#Visualization libraries:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

#Data processing:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

### Steps:

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Load dataset:
filepath = '/Workspace/Users/rogatorm@gmail.com/Patient_Readmission.xlsx'

df= pd.read_excel(filepath)
df.head()

,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
0,1,62,Female,Elective,4,5,110,130,1,No
1,2,65,Male,Emergency,19,2,157,81,4,No
2,3,82,Female,Emergency,18,4,74,84,0,No
3,4,85,Male,Emergency,2,4,106,85,4,No
4,5,85,Female,Elective,19,3,80,119,3,No


# Understanding Data:

In [0]:
print(df.shape)
print(df.info())

(3000, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Patient ID           3000 non-null   int64 
 1   Age                  3000 non-null   int64 
 2   Gender               3000 non-null   object
 3   Admission Type       3000 non-null   object
 4   Length of Stay       3000 non-null   int64 
 5   Number of Diagnoses  3000 non-null   int64 
 6   Blood Pressure       3000 non-null   int64 
 7   Blood Sugar Levels   3000 non-null   int64 
 8   Previous Admissions  3000 non-null   int64 
 9   Readmission          3000 non-null   object
dtypes: int64(7), object(3)
memory usage: 234.5+ KB
None


In [0]:
df.columns

Index(['Patient ID', 'Age', 'Gender', 'Admission Type', 'Length of Stay',
       'Number of Diagnoses', 'Blood Pressure', 'Blood Sugar Levels',
       'Previous Admissions', 'Readmission'],
      dtype='object')

### Dropping unneccessary columns:

In [0]:
df = df.drop('Patient ID', axis = 1)
df.head(3)

,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
0,62,Female,Elective,4,5,110,130,1,No
1,65,Male,Emergency,19,2,157,81,4,No
2,82,Female,Emergency,18,4,74,84,0,No


## Define Target and Features:

### Encoding:

In [0]:
# 2. Initialize the LabelEncoder
le = LabelEncoder()

# 3. Fit and transform the categorical data
df['Gender_encoded'] = le.fit_transform(df['Gender'])
df['Admission Type_encoded']=le.fit_transform(df['Admission Type'])
df['Readmission_encoded'] = le.fit_transform(df['Readmission'])

print(df)
print("\nCategory Mapping:", le.classes_)

      Age  Gender  ... Admission Type_encoded  Readmission_encoded
0      62  Female  ...                      0                    0
1      65    Male  ...                      1                    0
2      82  Female  ...                      1                    0
3      85    Male  ...                      1                    0
4      85  Female  ...                      0                    0
...   ...     ...  ...                    ...                  ...
2995   50  Female  ...                      1                    0
2996   21  Female  ...                      0                    0
2997   20  Female  ...                      0                    0
2998   69  Female  ...                      0                    1
2999   45  Female  ...                      0                    0

[3000 rows x 12 columns]

Category Mapping: ['No' 'Yes']


In [0]:
y = df['Readmission_encoded']

# 2. Define Features (drop all original text columns and the target columns)
X = df.drop(columns=[
    'Gender', 
    'Admission Type', 
    'Readmission', 
    'Readmission_encoded'
])

## Split, Scale then Fit - in this sequence always:

### Split:

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Scale:

In [0]:
# Initialize and apply StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Fit:

In [0]:
# 1. Initialize the Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# 2. Fit the model on the scaled training data
rf_model.fit(X_train_scaled, y_train)

# 3. Make predictions on the scaled test data
y_pred = rf_model.predict(X_test_scaled)

In [0]:
# Print evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.695

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.97      0.82       427
           1       0.14      0.01      0.02       173

    accuracy                           0.69       600
   macro avg       0.43      0.49      0.42       600
weighted avg       0.55      0.69      0.59       600

